# 05. 생성형 이미지: Stable Diffusion 직접 실행

문장을 넣고 **내 RunPod GPU에서 이미지 한 장**을 만든다. 앞 노트북의 변수나 파일은 필요 없다.
512×512 이미지부터 시작한다. 모델 다운로드에는 인터넷과 수 GB의 디스크가 필요하다.
GPU 여유 VRAM 8 GiB 이상을 시작점으로 권장하지만 이 노트북에서 실측한 최소 사양은 아니다.

사용 모델: [`stable-diffusion-v1-5/stable-diffusion-v1-5`](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5).
가중치 사용 조건은 모델 카드의 CreativeML OpenRAIL-M 라이선스를 확인한다.
이번 실습은 이미 학습된 가중치를 사용하는 **추론**이며 학습이나 fine-tuning은 하지 않는다.

## 준비

CUDA용 PyTorch와 Jupyter는 이미 설치되어 있다고 가정한다. 아래 추가 라이브러리가 없을 때만
Jupyter의 임시 코드 셀에서 다음 명령을 실행하고 **커널을 재시작**한다. 프로젝트 등록이나 공통 모듈은 없다.

```python
%pip install "diffusers==0.35.1" "transformers==4.56.2" "accelerate==1.10.1" "safetensors>=0.4.3" Pillow
```

위 버전은 API 기준을 고정한 것으로 RunPod GPU 실측 검증을 완료했다는 뜻은 아니다.
아래 셀을 위에서부터 실행한다. 다른 GPU 노트북의 커널은 종료하면 메모리를 확보할 수 있다.

In [ ]:
import gc
import json
import time
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version

import torch
from diffusers import StableDiffusionPipeline, DDIMScheduler
from huggingface_hub import model_info
from IPython.display import display

assert torch.cuda.is_available(), "CUDA 지원 PyTorch가 있는 GPU 커널을 선택하세요."
device = torch.device("cuda:0")
torch.cuda.set_device(device)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(device))
free, total = torch.cuda.mem_get_info(device)
print(f"VRAM free/total: {free / 2**30:.2f} / {total / 2**30:.2f} GiB")
print({name: version(name) for name in ["diffusers", "transformers", "accelerate"]})

# 노트북을 저장소 루트에서 연다. 두 경로는 기존 .gitignore에 포함되어 있다.
cache_dir = Path(".cache/huggingface")
output_dir = Path("runs/05_image_generation")
cache_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

## 1. 모델을 GPU에 올리기

첫 실행은 가중치를 다운로드하므로 오래 걸린다. 이후에는 캐시를 재사용한다.
`revision`은 실행 시점의 실제 커밋으로 고정하고 결과와 함께 저장한다.
나중에 동일 가중치를 사용하려면 저장된 revision 문자열을 직접 대입한다.

`float16`으로 가중치 메모리를 줄인다. VRAM이 부족하면 커널을 재시작하고
`cpu_offload = True`로 바꾼다. CPU offload는 CPU RAM과 전송 시간을 더 사용한다.
모델 로딩 셀을 반복 실행할 때 이전 모델이 남지 않도록 정리한다.

In [ ]:
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"
revision = model_info(model_id).sha  # 재현 시 metrics.json의 model_revision으로 교체
cpu_offload = False

if "pipe" in globals():
    del pipe
gc.collect()
torch.cuda.empty_cache()

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    revision=revision,
    cache_dir=str(cache_dir),
    torch_dtype=torch.float16,
    use_safetensors=True,
)
# DDIM을 사용해 step 수와 관찰 callback의 관계를 간단하게 만든다.
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
if cpu_offload:
    pipe.enable_model_cpu_offload(gpu_id=0)
else:
    pipe = pipe.to(device)
print("Model revision:", revision)
print("Scheduler:", type(pipe.scheduler).__name__)
print("UNet dtype:", pipe.unet.dtype)
print("UNet parameters:", sum(p.numel() for p in pipe.unet.parameters()))
print("Loaded allocated MiB:", torch.cuda.memory_allocated(device) / 2**20)

## 2. 생성 설정 직접 바꾸기

- `prompt`: 만들고 싶은 장면. 이 모델에서는 짧고 구체적인 영어로 먼저 실험한다.
- `negative_prompt`: 피하고 싶은 특성. guidance를 사용하는 경우에 적용된다.
- `seed`: 초기 노이즈를 정하는 값. 비교할 때 같은 seed를 쓴다.
- `steps`: 노이즈 제거 반복 수. 학습 epoch가 아니다.
- `guidance_scale`: 텍스트 조건의 영향. 크게 한다고 품질이 항상 좋아지지는 않는다.
- `width`, `height`: 처음에는 512로 두고, 8의 배수로 바꾼다. 해상도를 키우면 메모리도 증가한다.

흐름은 **텍스트 → CLIP embedding → latent 노이즈를 UNet과 scheduler로 반복 갱신 → VAE로 RGB 복원**이다.
512×512 출력의 latent는 보통 `[1, 4, 64, 64]`다. 아래에서 실제 shape를 확인한다.

In [ ]:
prompt = "a small robot reading a book in a cozy library, warm light, detailed digital art"
negative_prompt = "blurry, low quality, distorted, watermark, text"
seed = 42
steps = 25
guidance_scale = 7.5
width = 512
height = 512

assert width > 0 and height > 0 and width % 8 == 0 and height % 8 == 0
assert steps > 0

## 3. 생성 함수: 시간, VRAM, latent 관찰

매 호출마다 generator를 새로 만들어 seed를 다시 적용한다. 같은 generator 객체를 계속 쓰면 난수 상태가 진행된다.
callback에서는 이미지가 아닌 latent의 shape와 표준편차를 읽는다. 표준편차는 품질 점수가 아니며 단조 감소할 필요도 없다.
`observe=True`는 중간 텐서를 읽고 동기화하므로 속도 비교에서는 끈다.

타이머는 다운로드/모델 로딩/PNG 저장을 제외하고 텍스트 처리부터 최종 이미지 생성까지 포함한다.
CUDA 작업 완료를 기다린 뒤 시간을 읽는다. peak는 호출 직전에 초기화하지만 이미 로드된 모델의 메모리도 포함한다.

In [ ]:
def generate(prompt, negative_prompt, seed, steps, guidance_scale, width, height, observe=False):
    assert steps > 0 and width > 0 and height > 0 and width % 8 == 0 and height % 8 == 0
    observations = []

    def on_step_end(pipeline, step_index, timestep, callback_kwargs):
        if step_index == 0 or (step_index + 1) % 5 == 0 or step_index + 1 == steps:
            latents = callback_kwargs["latents"]
            row = {"step": step_index + 1, "timestep": int(timestep.item()),
                   "shape": list(latents.shape), "std": latents.float().std().item()}
            observations.append(row)
            print(row)
        return callback_kwargs  # latent를 수정하지 않고 반환

    kwargs = {}
    if observe:
        kwargs = {"callback_on_step_end": on_step_end,
                  "callback_on_step_end_tensor_inputs": ["latents"]}
    torch.cuda.synchronize(device)
    torch.cuda.reset_peak_memory_stats(device)
    start = time.perf_counter()
    with torch.inference_mode():
        result = pipe(
            prompt=prompt, negative_prompt=negative_prompt,
            generator=torch.Generator(device="cpu").manual_seed(seed),
            num_inference_steps=steps, guidance_scale=guidance_scale,
            width=width, height=height, **kwargs,
        )
    torch.cuda.synchronize(device)
    seconds = time.perf_counter() - start
    metrics = {
        "prompt": prompt, "negative_prompt": negative_prompt, "seed": seed,
        "steps": steps, "guidance_scale": guidance_scale, "width": width, "height": height,
        "seconds": seconds,
        "allocated_mib": torch.cuda.memory_allocated(device) / 2**20,
        "peak_allocated_mib": torch.cuda.max_memory_allocated(device) / 2**20,
        "reserved_mib": torch.cuda.memory_reserved(device) / 2**20,
        "model_id": model_id, "model_revision": revision,
        "scheduler": type(pipe.scheduler).__name__, "cpu_offload": cpu_offload,
        "gpu": torch.cuda.get_device_name(device), "torch": torch.__version__,
        "cuda": torch.version.cuda, "diffusers": version("diffusers"),
        "transformers": version("transformers"), "accelerate": version("accelerate"),
        "dtype": str(pipe.unet.dtype), "observations": observations,
        "nsfw_content_detected": result.nsfw_content_detected,
    }
    return result.images[0], metrics

## 4. 첫 이미지 생성하고 저장하기

이번 호출은 내부 관찰용이다. 첫 실행 초기화와 callback 비용이 포함되므로 성능 기준값으로 쓰지 않는다.
이미지는 셀 아래에 표시되고 PNG와 설정 JSON은 `runs/05_image_generation/`에 함께 저장된다.
모델의 기본 safety checker는 유지한다. 이미지가 검게 나오면 출력된 감지 결과도 확인한다.

In [ ]:
image, metrics = generate(prompt, negative_prompt, seed, steps, guidance_scale, width, height, observe=True)
display(image)
print(f"생성 시간: {metrics['seconds']:.2f}s, 최대 할당 VRAM: {metrics['peak_allocated_mib']:.1f} MiB")
print("Safety checker:", metrics["nsfw_content_detected"])

run_dir = output_dir / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
run_dir.mkdir(parents=True, exist_ok=False)
image.save(run_dir / "image.png")
(run_dir / "metrics.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장 위치:", run_dir.resolve())

## 5. 선택 실험: 같은 노이즈로 10 / 25 / 40 step 비교

아래 기본값은 `False`여서 Run All로 추가 생성하지 않는다. `True`로 바꾸면
짧은 warmup 후 이미지 세 장을 **순차 생성**한다. 반복 수만 바꾸고 나머지 조건은 고정한다.
배치 생성이나 학습 속도를 측정하는 코드가 아니다. 동일 seed라도 하드웨어/라이브러리 버전이 바뀌면 픽셀까지 같지는 않을 수 있다.

In [ ]:
run_comparison = False
if run_comparison:
    warmup_image, _ = generate(prompt, negative_prompt, seed, 2, guidance_scale, width, height)
    del warmup_image
    comparison_dir = output_dir / ("compare_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
    comparison_dir.mkdir(parents=True, exist_ok=False)
    rows = []
    for step_count in [10, 25, 40]:
        sample, row = generate(prompt, negative_prompt, seed, step_count, guidance_scale, width, height)
        display(sample)
        print(f"steps={step_count}, seconds={row['seconds']:.2f}, peak MiB={row['peak_allocated_mib']:.1f}")
        sample.save(comparison_dir / f"steps_{step_count}.png")
        rows.append(row)
    (comparison_dir / "metrics.json").write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding="utf-8")
    print("비교 결과:", comparison_dir.resolve())

## 직접 해볼 것 / 문제 해결

1. seed를 고정하고 `prompt`의 `warm light`를 `moonlight`로 바꾼 뒤 생성 셀을 다시 실행한다.
2. seed만 42 → 43으로 바꾼다. 구도와 세부가 얼마나 바뀌는가?
3. guidance를 1.0 / 7.5 / 12.0으로 비교한다. 1.0에서는 classifier-free guidance를 쓰지 않아 negative prompt의 영향도 없다.
4. 시간·이미지·VRAM을 함께 보고 step 수와 해상도 중 무엇을 먼저 늘릴지 생각해 본다.

- OOM: 다른 커널을 종료하고 512×512 한 장으로 시작한다. 필요하면 커널 재시작 후 `cpu_offload=True`를 사용한다.
- import 오류: 준비 셀의 라이브러리를 현재 커널에 설치했는지 확인하고 재시작한다.
- 다운로드 실패: 네트워크와 디스크 여유를 확인한다. 401/403이면 해당 모델의 접근 조건을 확인한다. 토큰을 노트북에 적지 않는다.
- `reserved`가 크다고 모두 살아 있는 텐서는 아니다. `nvidia-smi`는 CUDA context 등도 포함하므로 PyTorch 수치와 다르다.
- 캐시와 PNG/JSON은 Git 제외 경로에 있다. **노트북 자체의 출력 이미지는 Git에 포함될 수 있으므로 커밋 전에 Clear All Outputs 후 저장한다.**

실습을 끝내고 GPU 메모리를 반환하려면 아래 셀을 실행한다. 다시 생성하려면 모델 로딩 셀부터 실행한다.

In [ ]:
release_gpu = False  # 실습을 끝냈을 때 True로 변경
if release_gpu:
    del pipe
    gc.collect()
    torch.cuda.empty_cache()
    print("파이프라인 해제 완료. 다른 커널이 사용하는 VRAM은 별도입니다.")

## 참고 및 검증 범위

- [Diffusers 0.35.1 StableDiffusionPipeline API](https://huggingface.co/docs/diffusers/v0.35.1/api/pipelines/stable_diffusion/text2img)
- [모델 카드와 라이선스](https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5)

노트북 구조와 Python 셀의 문법은 확인했다. **실제 RunPod GPU에서 모델 다운로드·이미지 생성·VRAM 측정은 아직 검증하지 않았다.**
출력은 의도적으로 비워 두었으며 예시 생성 결과를 실측처럼 첨부하지 않았다.